# Блокнот 19. Ускорение и деплой: замеры, ONNX, квантование, OpenVINO, TensorRT и трекинг

Курс «Машинное обучение с нуля», модуль 19. Разделы: 1 — урок 19.1, 2 — урок 19.2, 3 — урок 19.3, 4 — урок 19.4, 5 — урок 19.5, 6 — урок 19.6. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

Блокнот рассчитан на процессор: ускорять модель для ноутбука интереснее всего именно там. Только вторая половина раздела 5 (TensorRT) требует видеокарту NVIDIA — в Colab включите её: «Среда выполнения → Сменить среду выполнения → T4 GPU». Без видеокарты эта ячейка просто пропускается.

Блокнот ставит onnx, onnxruntime, openvino и nncf (≈150 МБ), скачивает Fashion-MNIST (≈30 МБ), картинку COCO (≈170 КБ), веса YOLO26n (≈5 МБ) и учебный набор coco8 (≈1 МБ).

In [ ]:
%pip install -q ultralytics onnx onnxruntime onnxscript openvino nncf lap

In [ ]:
import copy
import logging
import os
import time
import urllib.request

import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnx
import onnxruntime as ort
import openvino as ov
import torch
from torch import nn
from torchvision import datasets
from ultralytics import YOLO

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
DATA = os.environ.get("COURSE_DATA", "data")
os.makedirs(DATA, exist_ok=True)
print("устройство:", device, "| torch", torch.__version__, "| onnxruntime", ort.__version__, "| openvino", ov.__version__,
      "| opencv", cv2.__version__)

img_path = os.path.join(DATA, "coco_000000039769.jpg")
if not os.path.exists(img_path):
    urllib.request.urlretrieve("http://images.cocodataset.org/val2017/000000039769.jpg", img_path)
frame = cv2.imread(img_path)  # кадр 480 × 640, каналы BGR — как с камеры

## 1. Где теряется время (урок 19.1)

Первый вызов модели всегда медленный: загрузка весов, выделение памяти, подготовка вычислений. Поэтому мерят **после прогрева** и много раз подряд, а итог описывают медианой и 95-м перцентилем.

**Задание 1.** Допишите `measure(fn, warmup=3, runs=20)`: вызовите `fn()` `warmup` раз без замера, затем `runs` раз с замером каждого вызова через `time.perf_counter()` и верните массив NumPy с временами в **миллисекундах**.

In [ ]:
def measure(fn, warmup=3, runs=20):
    ...  # ваш код


yolo = YOLO("yolo26n.pt")
t0 = time.perf_counter()
yolo.predict(frame, verbose=False)
print(f"первый вызов: {(time.perf_counter() - t0) * 1000:.0f} мс")
t = measure(lambda: yolo.predict(frame, verbose=False))
assert t.shape == (20,) and t.min() > 0
print(f"после прогрева: медиана {np.median(t):.1f} мс, p95 {np.percentile(t, 95):.1f} мс → "
      f"{1000 / np.median(t):.0f} кадров в секунду")

Разложим кадр на этапы. Ultralytics сам пишет в `result.speed`, сколько миллисекунд ушло на предобработку (letterbox, нормализация), саму сеть и постобработку; чтение файла и рисование рамок замерим сами.

In [ ]:
speeds = [yolo.predict(frame, verbose=False)[0].speed for _ in range(20)]
r = yolo.predict(frame, verbose=False)[0]
stages = {
    "чтение кадра": np.median(measure(lambda: cv2.imread(img_path))),
    "предобработка": np.median([s["preprocess"] for s in speeds]),
    "сеть": np.median([s["inference"] for s in speeds]),
    "постобработка": np.median([s["postprocess"] for s in speeds]),
    "рисование рамок": np.median(measure(lambda: r.plot())),
}
total = sum(stages.values())
for name, ms in stages.items():
    print(f"{name:16s} {ms:6.1f} мс  {ms / total:4.0%}")
print(f"{'всего':16s} {total:6.1f} мс → {1000 / total:.0f} кадров в секунду")

Теперь батч. Подадим сети сразу `b` картинок 640 × 640 и посмотрим, как меняются время на батч (задержка) и число картинок в секунду (пропускная способность). Берём копию сети со слитыми слоями — так её применяет `predict` (урок 19.5).

In [ ]:
net = copy.deepcopy(yolo.model).fuse(verbose=False).eval()
rows = []
with torch.inference_mode():
    for b in (1, 2, 4, 8):
        x = torch.rand(b, 3, 640, 640)
        ms = np.median(measure(lambda: net(x), warmup=2, runs=5))
        rows.append((b, ms, b * 1000 / ms))
        print(f"батч {b}: {ms:6.0f} мс на батч, {b * 1000 / ms:5.1f} картинок в секунду")
fig, ax1 = plt.subplots(figsize=(6, 3.5))
bs, lat, thr = zip(*rows)
ax1.plot(bs, lat, "o-", color="tab:orange")
ax1.set_xlabel("размер батча")
ax1.set_ylabel("задержка, мс", color="tab:orange")
ax2 = ax1.twinx()
ax2.plot(bs, thr, "s-", color="tab:blue")
ax2.set_ylabel("картинок в секунду", color="tab:blue")
ax2.set_ylim(0, max(thr) * 1.3)
plt.title("YOLO26n на процессоре: батч и скорость")
plt.show()

На процессоре батч прибавляет немного картинок в секунду — ядра и так почти заняты одной картинкой, а задержка растёт пропорционально батчу. На видеокарте батч даёт большой выигрыш в пропускной способности. Для камеры в реальном времени батч — 1.

## 2. ONNX: экспорт и ONNX Runtime (урок 19.2)

Возьмём маленькую CNN из урока 13.3 и обучим её 2 эпохи на Fashion-MNIST (≈30 с на процессоре) — она пригодится для экспорта и квантования.

In [ ]:
train = datasets.FashionMNIST(DATA, train=True, download=True)
test = datasets.FashionMNIST(DATA, train=False, download=True)
MEAN, STD = 0.2860, 0.3530
X_tr = ((train.data.float() / 255 - MEAN) / STD).unsqueeze(1)  # (60000, 1, 28, 28)
X_te = ((test.data.float() / 255 - MEAN) / STD).unsqueeze(1)
y_tr, y_te = train.targets, test.targets

cnn = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10))
opt = torch.optim.AdamW(cnn.parameters(), lr=1e-3)
for epoch in range(2):
    perm = torch.randperm(len(X_tr))
    for i in range(0, len(perm), 128):
        b = perm[i:i + 128]
        opt.zero_grad()
        nn.functional.cross_entropy(cnn(X_tr[b]), y_tr[b]).backward()
        opt.step()
cnn.eval()  # обязательно перед экспортом
with torch.no_grad():
    torch_logits = cnn(X_te).numpy()
acc_torch = (torch_logits.argmax(1) == y_te.numpy()).mean()
print(f"точность PyTorch: {acc_torch:.4f}")

Экспорт: PyTorch трассирует модель на примере входа и записывает граф операторов и веса в файл. Размер батча делаем **динамическим**, чтобы файл принимал любой батч.

In [ ]:
x = torch.randn(1, 1, 28, 28)
torch.onnx.export(cnn, (x,), "cnn.onnx", input_names=["image"], output_names=["logits"],
                  dynamic_shapes=({0: torch.export.Dim("batch")},), external_data=False)
m = onnx.load("cnn.onnx")
onnx.checker.check_model(m)
print("opset:", m.opset_import[0].version)
print("операторы:", [n.op_type for n in m.graph.node])
print("вход:", m.graph.input[0].name, [d.dim_param or d.dim_value for d in m.graph.input[0].type.tensor_type.shape.dim])
print(f"размер файла: {os.path.getsize('cnn.onnx') / 1e6:.2f} МБ")

Запускаем файл в ONNX Runtime — без PyTorch. Выходы должны совпасть с PyTorch до ошибок округления.

In [ ]:
sess = ort.InferenceSession("cnn.onnx", providers=["CPUExecutionProvider"])
ort_logits = sess.run(None, {"image": X_te.numpy()})[0]
diff = np.abs(ort_logits - torch_logits).max()
print(f"наибольшая разница с PyTorch: {diff:.2e}")
np.testing.assert_allclose(ort_logits, torch_logits, rtol=1e-3, atol=1e-4)
acc_ort = (ort_logits.argmax(1) == y_te.numpy()).mean()
print(f"точность ONNX Runtime: {acc_ort:.4f}")

**Задание 2.** В приложении на вход придёт картинка как массив `uint8` 28 × 28 (числа 0–255), а не готовый тензор. Допишите `preprocess(img)`: верните массив `float32` формы `(1, 1, 28, 28)` с **той же** нормализацией, что при обучении: `(img / 255 − MEAN) / STD`.

In [ ]:
def preprocess(img):
    ...  # ваш код


raw = test.data.numpy()  # картинки как есть: uint8 0–255
p = preprocess(raw[0])
assert p.shape == (1, 1, 28, 28) and p.dtype == np.float32
np.testing.assert_allclose(p, X_te[:1].numpy(), atol=1e-6)
pred = [int(sess.run(None, {"image": preprocess(raw[i])})[0].argmax()) for i in range(1000)]
print(f"точность на 1000 «сырых» картинках: {(np.array(pred) == y_te[:1000].numpy()).mean():.3f}")

# частая ошибка: забыли нормализацию — модель видит не те числа
bad = sess.run(None, {"image": (raw[:1000, None] / 255).astype(np.float32)})[0]
print(f"без нормализации: {(bad.argmax(1) == y_te[:1000].numpy()).mean():.3f}")

ONNX Runtime часто быстрее PyTorch на процессоре: он заранее оптимизирует граф (сливает операции, выкидывает лишнее). Сравним на одной картинке и на батче 256.

In [ ]:
cnn_speed = {}
for b in (1, 256):
    xb = X_te[:b]
    with torch.inference_mode():
        t_torch = np.median(measure(lambda: cnn(xb), runs=30))
    t_ort = np.median(measure(lambda: sess.run(None, {"image": xb.numpy()}), runs=30))
    cnn_speed[b] = (t_torch, t_ort)
    print(f"батч {b:3d}: PyTorch {t_torch:6.2f} мс, ONNX Runtime {t_ort:6.2f} мс")

YOLO экспортируется одной командой. По умолчанию Ultralytics записывает сырой выход сети (1 × 84 × 8400, как у YOLO11 в уроке 17.9) и выполняет NMS сам. С `nms=False` YOLO26 экспортируется с головой без NMS (урок 17.6): выход файла — сразу 300 строк «рамка, уверенность, класс».

In [ ]:
onnx_path = yolo.export(format="onnx", nms=False, verbose=False)
yolo_onnx = YOLO(onnx_path, task="detect")
s = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
print("вход:", s.get_inputs()[0].shape, "→ выход:", s.get_outputs()[0].shape)
for name, mdl in (("PyTorch", yolo), ("ONNX Runtime", yolo_onnx)):
    r = mdl.predict(frame, verbose=False)[0]
    t = np.median(measure(lambda: mdl.predict(frame, verbose=False)))
    print(f"{name:13s} {t:5.1f} мс: {[r.names[int(c)] for c in r.boxes.cls]}")

## 3. Квантование и прунинг (урок 19.3)

**Задание 3.** Допишите `quantize_sym(w)` — симметричное квантование в INT8, как в уроке 19.3: масштаб `s = max|w| / 127`, `q = round(w / s)` (тип `torch.int8`). Верните `q` и `s`.

In [ ]:
def quantize_sym(w):
    ...  # ваш код


w = cnn[0].weight.detach()
q, s = quantize_sym(w)
assert q.dtype == torch.int8 and q.abs().max() == 127
err = (q.float() * s - w).abs().max()
print(f"веса первой свёртки: {w.numel()} чисел, масштаб s = {s:.5f}, наибольшая ошибка {err:.5f} (не больше s / 2)")
assert err <= s / 2 + 1e-6

n_params = sum(p.numel() for p in cnn.parameters())
print(f"параметров {n_params}: FP32 — {n_params * 4 / 1e6:.2f} МБ, INT8 — {n_params / 1e6:.2f} МБ")

Квантуем всю сеть средствами ONNX Runtime. Веса квантуются сразу, а для активаций нужна **калибровка**: прогоняем 300 обучающих картинок и запоминаем диапазоны значений на каждом слое.

In [ ]:
from onnxruntime.quantization import CalibrationDataReader, QuantFormat, QuantType, quantize_static


class Calibration(CalibrationDataReader):
    def __init__(self, X):
        self.batches = iter([{"image": X[i:i + 1].numpy()} for i in range(len(X))])

    def get_next(self):
        return next(self.batches, None)


quantize_static("cnn.onnx", "cnn_int8.onnx", Calibration(X_tr[:300]), quant_format=QuantFormat.QDQ,
                per_channel=True, weight_type=QuantType.QInt8, activation_type=QuantType.QInt8)
sess8 = ort.InferenceSession("cnn_int8.onnx", providers=["CPUExecutionProvider"])
acc8 = (sess8.run(None, {"image": X_te.numpy()})[0].argmax(1) == y_te.numpy()).mean()
t8 = np.median(measure(lambda: sess8.run(None, {"image": X_te[:256].numpy()}), runs=30))
print(f"FP32: {os.path.getsize('cnn.onnx') / 1e6:.2f} МБ, точность {acc_ort:.4f}, батч 256 — {cnn_speed[256][1]:.2f} мс")
print(f"INT8: {os.path.getsize('cnn_int8.onnx') / 1e6:.2f} МБ, точность {acc8:.4f}, батч 256 — {t8:.2f} мс")

Прунинг без дообучения: обнулим долю самых маленьких по модулю весов во всех слоях сразу и посмотрим на точность.

In [ ]:
import copy

from torch.nn.utils import prune

for amount in (0.5, 0.8, 0.9, 0.95):
    pruned = copy.deepcopy(cnn)
    params = [(mod, "weight") for mod in pruned if isinstance(mod, (nn.Conv2d, nn.Linear))]
    prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=amount)
    with torch.no_grad():
        acc_p = (pruned(X_te).argmax(1) == y_te).float().mean().item()
    print(f"обнулено {amount:.0%} весов: точность {acc_p:.4f}")

Половину весов можно выбросить почти даром, но от нулей вразброс сеть не стала ни меньше, ни быстрее: матрицы того же размера. Ускоряет только структурированный прунинг (целые каналы) или специальное железо (урок 19.3).

## 4. OpenVINO и NNCF (урок 19.4)

In [ ]:
core = ov.Core()
print("устройства OpenVINO:", core.available_devices)
ov_model = ov.convert_model("cnn.onnx")
ov_cnn = core.compile_model(ov_model, "CPU")
ov_logits = ov_cnn(X_te.numpy())[0]
print(f"разница с PyTorch: {np.abs(ov_logits - torch_logits).max():.2e}, "
      f"точность {(ov_logits.argmax(1) == y_te.numpy()).mean():.4f}")

NNCF квантует модель OpenVINO по калибровочному набору — тем же 300 картинкам. Функция преобразования `transform_fn` достаёт из элемента набора то, что подаётся на вход модели.

In [ ]:
import nncf

logging.getLogger("nncf").setLevel(logging.ERROR)  # без подробного журнала квантования
calib = nncf.Dataset([X_tr[i:i + 1].numpy() for i in range(300)], lambda item: item)
ov_int8 = core.compile_model(nncf.quantize(ov_model, calib, subset_size=300), "CPU")
acc_ov8 = (ov_int8(X_te.numpy())[0].argmax(1) == y_te.numpy()).mean()
for name, mdl in (("OpenVINO FP32", ov_cnn), ("OpenVINO INT8", ov_int8)):
    t = np.median(measure(lambda: mdl(X_te[:256].numpy()), runs=30))
    print(f"{name}: батч 256 — {t:.2f} мс")
print(f"точность INT8: {acc_ov8:.4f}")

Теперь YOLO26n. Ultralytics делает OpenVINO-модель одной командой, а с `quantize=8` ещё и квантует её через NNCF; для калибровки нужен набор — возьмём учебный coco8. Сравним все варианты на одной картинке.

In [ ]:
ov_path = yolo.export(format="openvino", nms=False, verbose=False)
ov8_path = yolo.export(format="openvino", nms=False, quantize=8, data="coco8.yaml", verbose=False)
variants = {"PyTorch": yolo, "ONNX Runtime": yolo_onnx, "OpenVINO FP32": YOLO(ov_path, task="detect"),
            "OpenVINO INT8": YOLO(ov8_path, task="detect")}
yolo_speed = {}
for name, mdl in variants.items():
    r = mdl.predict(frame, verbose=False)[0]
    yolo_speed[name] = np.median([mdl.predict(frame, verbose=False)[0].speed["inference"] for _ in range(20)])
    print(f"{name:14s} сеть {yolo_speed[name]:5.1f} мс: {[r.names[int(c)] for c in r.boxes.cls]}")

plt.figure(figsize=(6, 3))
plt.barh(list(yolo_speed)[::-1], list(yolo_speed.values())[::-1], color="tab:blue")
plt.xlabel("время сети на кадр, мс (меньше — лучше)")
plt.title("YOLO26n на процессоре")
plt.show()

Калибровка по 4 картинкам coco8 — только для демонстрации: в настоящем проекте берут 300–500 картинок из своих данных и проверяют mAP до и после квантования.

## 5. Слияние слоёв и TensorRT (урок 19.5)

Первое, что делает TensorRT (и OpenVINO, и ONNX Runtime) с графом, — сливает слои. BatchNorm после свёртки в режиме инференса — это умножение на число и сдвиг для каждого канала, и их можно «впечатать» в веса свёртки.

**Задание 4.** Допишите `fold_bn(conv, bn)`: верните новую свёртку с весами $w' = w \cdot \gamma / \sqrt{\sigma^2 + \varepsilon}$ (для каждого выходного канала) и смещением $b' = (b - \mu) \cdot \gamma / \sqrt{\sigma^2 + \varepsilon} + \beta$. Здесь $\gamma$ — `bn.weight`, $\beta$ — `bn.bias`, $\mu$ — `bn.running_mean`, $\sigma^2$ — `bn.running_var`, $\varepsilon$ — `bn.eps`.

In [ ]:
def fold_bn(conv, bn):
    fused = nn.Conv2d(conv.in_channels, conv.out_channels, conv.kernel_size, conv.stride, conv.padding, bias=True)
    with torch.no_grad():
        ...  # ваш код
    return fused


conv, bn = nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16)
with torch.no_grad():  # статистики, как у обученной сети
    bn.running_mean.uniform_(-1, 1)
    bn.running_var.uniform_(0.5, 2)
    bn.weight.uniform_(0.5, 1.5)
    bn.bias.uniform_(-0.5, 0.5)
pair = nn.Sequential(conv, bn).eval()
fused = fold_bn(conv, bn).eval()
x = torch.randn(8, 3, 64, 64)
with torch.inference_mode():
    print(f"разница выходов: {(pair(x) - fused(x)).abs().max():.2e}")
    assert torch.allclose(pair(x), fused(x), atol=1e-5)

Ответы те же, а слоёв меньше. Ultralytics делает такое слияние перед инференсом (`model.fuse()`). Сравним сеть YOLO26n до и после него на картинке 640 × 640.

In [ ]:
unfused = yolo.model.eval()
fused_yolo = copy.deepcopy(unfused).fuse(verbose=False).eval()
x = torch.rand(1, 3, 640, 640)
with torch.inference_mode():
    print(f"параметров: {sum(p.numel() for p in unfused.parameters())} → {sum(p.numel() for p in fused_yolo.parameters())}")
    print(f"без слияния {np.median(measure(lambda: unfused(x))):.1f} мс, "
          f"со слиянием {np.median(measure(lambda: fused_yolo(x))):.1f} мс")

TensorRT нужна видеокарта NVIDIA: следующая ячейка собирает engine для YOLO26n в FP16 и сравнивает скорость с PyTorch. Без видеокарты она пропускается.

In [ ]:
if device == "cuda":
    engine_path = yolo.export(format="engine", quantize=16, verbose=False)
    trt_yolo = YOLO(engine_path, task="detect")
    for name, mdl in (("PyTorch GPU", yolo), ("TensorRT FP16", trt_yolo)):
        r = mdl.predict(frame, verbose=False, device=0)[0]
        t = np.median([mdl.predict(frame, verbose=False, device=0)[0].speed["inference"] for _ in range(50)])
        print(f"{name:14s} сеть {t:5.2f} мс: {[r.names[int(c)] for c in r.boxes.cls]}")
else:
    print("Видеокарты NVIDIA нет — TensorRT пропускаем. В Colab: «Среда выполнения → T4 GPU» и запустите блокнот заново.")

## 6. Видеопоток и трекинг (урок 19.6)

Сделаем два коротких «видео» сами: фото с кошками медленно едет справа налево по широкому кадру цвета дивана. Во втором видео посередине стоит серый столб, за которым кошки по очереди проходят. Запишем оба в файлы, как настоящую съёмку.

In [ ]:
H, W, N, LINE = 480, 1120, 48, 560  # высота и ширина кадра, число кадров, x середины кадра
bg = np.median(frame[420:480, 250:450].reshape(-1, 3), axis=0).astype(np.uint8)  # цвет дивана


def make_video(path, pole):
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), 12, (W, H))
    for k in range(N):
        dx = round(480 - k * 480 / (N - 1))  # фото сдвигается влево на 10 пикселей за кадр
        f = np.full((H, W, 3), bg, np.uint8)
        f[:, dx:dx + 640] = frame
        if pole:
            f[:, LINE - 20:LINE + 20] = (128, 128, 128)  # столб шириной 40 пикселей
        writer.write(f)
    writer.release()


make_video("cats.mp4", pole=False)
make_video("cats_pole.mp4", pole=True)
print(f"записано по {N} кадров {W} × {H}")

Читаем видео кадр за кадром через `cv2.VideoCapture` и отдаём каждый кадр трекеру ByteTrack. `persist=True` говорит Ultralytics, что кадр — продолжение того же потока и номера треков надо сохранять. Для каждого трека запоминаем центры рамок.

In [ ]:
def track_video(path, tracker="bytetrack.yaml"):
    model = YOLO("yolo26n.pt")  # новая модель — новый трекер с нуля
    cap = cv2.VideoCapture(path)
    history = {}  # номер трека → список (кадр, x центра, y центра)
    k, last = 0, None
    while True:
        ok, f = cap.read()
        if not ok:
            break
        r = model.track(f, persist=True, tracker=tracker, classes=[15], verbose=False)[0]  # 15 — кошки
        if r.boxes.id is not None:
            for tid, (cx, cy, w_, h_) in zip(r.boxes.id.int().tolist(), r.boxes.xywh.tolist()):
                history.setdefault(tid, []).append((k, cx, cy))
        k, last = k + 1, f
    cap.release()
    return history, last


history, last = track_video("cats.mp4")
print("треки:", {tid: len(pts) for tid, pts in history.items()}, "(сколько кадров видели каждый трек)")

plt.figure(figsize=(9, 4))
plt.imshow(last[..., ::-1])
for tid, pts in history.items():
    _, xs, ys = zip(*pts)
    plt.plot(xs, ys, ".-", label=f"трек {tid}")
plt.axvline(LINE, color="yellow", ls="--")
plt.legend()
plt.title("Центры рамок по кадрам: кошки едут влево через линию")
plt.axis("off")
plt.show()

Две длинные траектории — две кошки. Если у вас есть ещё трек длиной в 1–2 кадра, это ложное срабатывание детектора: такие короткие треки обычно отбрасывают по длине.

**Задание 5.** Посчитаем, сколько кошек пересекло вертикальную линию `x = LINE` справа налево. Допишите `count_crossings(history, x_line)`: трек засчитывается, если у него есть два **соседних** замера, где центр был правее линии (`x > x_line`), а стал левее или на ней (`x <= x_line`).

In [ ]:
def count_crossings(history, x_line):
    ...  # ваш код


assert count_crossings({1: [(0, 250, 0), (1, 210, 0), (2, 190, 0)], 2: [(0, 100, 0), (1, 90, 0)]}, 200) == 1
assert count_crossings({1: [(0, 150, 0), (1, 260, 0)]}, 200) == 0
print("кошек пересекло линию:", count_crossings(history, LINE))

Теперь видео со столбом. Пока кошка за столбом, детектор видит её хуже: уверенность падает, а иногда кошка распадается на две рамки — по обе стороны столба.

In [ ]:
history_pole, last_pole = track_video("cats_pole.mp4")
print("треки:", {tid: len(pts) for tid, pts in history_pole.items()})
print("кошек пересекло линию:", count_crossings(history_pole, LINE))

У автора курса со столбом получилось 4 трека на двух кошек: короткий трек у «половинки» кошки и смена номера у одной из кошек. А пересечений линии — по-прежнему 2: счёт по линии устойчивее, чем число треков. Попробуйте трекер по умолчанию (`tracker` не указывать — это TrackTrack) и `botsort.yaml`: у каждого свои ошибки.

### OpenCV DNN без Ultralytics

Модуль `cv2.dnn` читает ONNX-файл сам. Кадр превращается в тензор функцией `cv2.dnn.blobFromImage`: масштаб 1/255, размер 640 × 640 (растяжение, без letterbox), BGR → RGB. Рамки пересчитываем обратно отдельно по ширине и высоте. Ultralytics при импорте выключает многопоточность OpenCV (`cv2.setNumThreads(0)`) — вернём её, иначе `cv2.dnn` будет считать в один поток.

In [ ]:
cv2.setNumThreads(-1)  # -1 — число потоков по умолчанию
try:
    cv_net = cv2.dnn.readNetFromONNX(os.path.basename(onnx_path))  # файл в текущей папке
    blob = cv2.dnn.blobFromImage(frame, 1 / 255, (640, 640), swapRB=True)
    cv_net.setInput(blob)
    out = cv_net.forward()  # (1, 300, 6): x1, y1, x2, y2 в пикселях 640 × 640, уверенность, класс
    sx, sy = frame.shape[1] / 640, frame.shape[0] / 640
    for x1, y1, x2, y2, conf, cls in out[0][out[0][:, 4] > 0.5]:
        print(f"{yolo.names[int(cls)]:8s} {conf:.2f}  рамка {x1 * sx:.0f}, {y1 * sy:.0f}, {x2 * sx:.0f}, {y2 * sy:.0f}")

    def cv_forward():
        cv_net.setInput(blob)
        cv_net.forward()

    print(f"OpenCV {cv2.__version__}: {np.median(measure(cv_forward)):.1f} мс на кадр 640 × 640")
except cv2.error as e:
    print("OpenCV", cv2.__version__, "не смог прочитать эту модель:", str(e).splitlines()[-1][:120])
    print("Запускайте такие модели в ONNX Runtime, а OpenCV оставьте для видео и рисования.")

Тот же цикл чтения работает с веб-камерой (`cv2.VideoCapture(0)`) и с сетевой камерой (`"rtsp://…"`). В Colab камеры нет, поэтому мы и сделали видео сами.